In [1]:
import sys; sys.path.append("../src")
import torch
from data_prep import prepare_data
from models import IDSNet
from attacks import fgsm

torch.manual_seed(42)
d = prepare_data("../data/KDDTrain.txt", "../data/KDDTest.txt")
X_val = torch.tensor(d["X_val"]); y_val = torch.tensor(d["y_val"], dtype=torch.long)

model = IDSNet(X_val.shape[1])
model.load_state_dict(torch.load("../saved_models/nn_baseline.pt"))
model.eval()

IDSNet(
  (net): Sequential(
    (0): Linear(in_features=121, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)

In [2]:
# Keep only real attacks that the model currently catches
with torch.no_grad():
    caught = model(X_val).argmax(dim=1) == 1
mask = (y_val == 1) & caught
X_atk, y_atk = X_val[mask], y_val[mask]
print("attacks the model catches:", len(X_atk))

attacks the model catches: 11673


In [3]:
print(f"{'eps':>5} | {'FGSM evasion':>13} | {'random noise evasion':>20}")
for eps in [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]:
    X_adv = fgsm(model, X_atk, y_atk, eps)
    X_rand = X_atk + eps * torch.sign(torch.randn_like(X_atk))
    with torch.no_grad():
        fgsm_evaded = (model(X_adv).argmax(dim=1) == 0).float().mean().item()
        rand_evaded = (model(X_rand).argmax(dim=1) == 0).float().mean().item()
    print(f"{eps:>5.2f} | {fgsm_evaded*100:>12.1f}% | {rand_evaded*100:>19.1f}%")

  eps |  FGSM evasion | random noise evasion
 0.00 |          0.0% |                 0.0%
 0.05 |         27.8% |                 0.2%
 0.10 |         97.1% |                 0.8%
 0.20 |         99.6% |                 3.2%
 0.50 |        100.0% |                19.3%
 1.00 |         99.9% |                33.5%
